### Leer todos los datos que son requeridos

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date","2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

movies_df = spark.read.table("movie_silver.movies") \
                        .filter(f"file_date = '{v_file_date}'")  
genres_df = spark.read.table("movie_silver.genres")
movies_genres_df = spark.read.table("movie_silver.movies_genres") \
                        .filter(f"file_date = '{v_file_date}'")  

In [0]:
window_spec = (
    Window
    .partitionBy("year_release_date")
    .orderBy(
        F.desc("total_budget"),
        F.desc("total_revenue")
    )
)

In [0]:
result_df = (
    movies_df.alias("m")
    .filter(F.col("year_release_date") >= 2015)
    .join(
        movies_genres_df.alias("mg"),
        F.col("m.movie_id") == F.col("mg.movie_id"),
        "inner"
    )
    .join(
        genres_df.alias("g"),
        F.col("mg.genre_id") == F.col("g.genre_id"),
        "inner"
    )
    .groupBy(
        "year_release_date",
        "genre_name"
    )
    .agg(
        F.sum("budget").alias("total_budget"),
        F.sum("revenue").alias("total_revenue")
    )
    .withColumn("dense_rank", F.dense_rank().over(window_spec))
    .withColumn("created_date",F.lit(v_file_date))
)

In [0]:
#overwrite_partition("movie_gold","results_group_movie_genre","created_date", v_file_date)

In [0]:
merge_condition = "tgt.year_release_date = src.year_release_date AND tgt.genre_name = src.genre_name AND tgt.created_date = src.created_date"
merge_delta_lake(result_df,"movie_gold","results_group_movie_genre",merge_condition,"created_date")


In [0]:
%sql
SELECT created_date,
COUNT(1) 
FROM movie_gold.results_group_movie_genre
GROUP BY created_date

created_date,count(1)
2024-12-23,16
2024-12-30,38
